# Stage 5 Sensitivity Models — DEC119 v1.3 COMMITTEE-AUDITED DRIVE-SAFE

**Purpose:** run the pre-specified sensitivity/supportive analyses after the Stage 5 primary C-minimal models.

This notebook **does not replace the primary model**. The primary model remains:

`clinical baseline + EF + E/e′ septal`.

This notebook examines whether the primary story is stable when adding broader clinically motivated echo variables:

- **C-lite:** C-minimal + SPAP + LA size
- **C-full:** C-lite + TR
- **No-albumin sensitivity:** checks whether albumin absorbs part of the echo signal
- **Hospitalization model sensitivities:** NB, robust Poisson, ZINB; full cohort remains the locked primary estimand and ≥6m remains mandatory sensitivity
- **Cox PH sensitivity:** regular Cox + PH diagnostics + albumin time-varying sensitivity for broader tiers

**Interpretation rule:** all C-lite/C-full/no-albumin results are supportive/exploratory. They should not be used to redefine the primary model after seeing the data.

## Committee update before running DEC119 v1.1

The Stage 5 v1.8.1 outputs suggested a time-dependent pattern:

- E/e′ septal did **not** add to 1-year mortality.
- E/e′ septal did add to long-term survival and hospitalization burden.
- Landmark outputs suggested E/e′ septal may be more relevant after 6 months, while albumin may be more important early.

This notebook therefore adds a **carry-forward reporting audit** from the Stage 5 v1.8.1 outputs if those files exist in Drive. This does **not** change the locked primary model or the hospitalization estimand. It only records how the sensitivity analyses should be interpreted.

**Hospitalization reporting rule remains committee-consistent:** full cohort is the locked primary cohort; ≥6m cohort is a mandatory sensitivity. If full-cohort NB is flagged, both full and ≥6m results should be shown side-by-side, and the 6m result should be used to support robustness, not to silently replace the primary estimand.


## Drive-safe update in v1.2

This version writes all CSV outputs to local Colab storage first (`/content/...`) and only copies a ZIP to Google Drive at the end. This reduces `ConnectionAbortedError` errors caused by repeated writes to Drive. It also uses robust input-file discovery with the same candidate filenames used in earlier notebooks.


In [1]:
# ============================================================
# 0. Environment setup — DRIVE-SAFE VERSION
# ============================================================
import sys, os, json, math, warnings, subprocess, importlib.util, shutil
from pathlib import Path
from datetime import datetime

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    # Safe mount: do not fail if Drive is already mounted in this runtime.
    if Path('/content/drive/MyDrive').exists():
        print('Google Drive already mounted.')
    else:
        drive.mount('/content/drive')

# lifelines is usually not preinstalled in Colab.
if importlib.util.find_spec('lifelines') is None:
    print('Installing lifelines...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines'])

import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from sklearn.metrics import roc_auc_score
from lifelines import CoxPHFitter, CoxTimeVaryingFitter
from lifelines.statistics import proportional_hazard_test

warnings.filterwarnings('default')
pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 220)

RUN_VERSION = 'DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE'
RUN_TS = datetime.now().isoformat(timespec='seconds')
print('Run version:', RUN_VERSION)
print('Run timestamp:', RUN_TS)


Mounted at /content/drive
Installing lifelines...
Run version: DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE
Run timestamp: 2026-05-06T18:51:37


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# ============================================================
# 1. Paths — write locally first, copy ZIP to Drive at the end
# ============================================================
if IN_COLAB:
    PREFERRED_STAGE1_DIR = Path('/content/drive/MyDrive/dialysis/outputs/params/stage1')
    # Most writes go to local Colab storage to avoid Google Drive connection-abort errors.
    OUTPUT_DIR = Path('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE')
    DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE')
else:
    PREFERRED_STAGE1_DIR = Path('/mnt/data/stage1_uploaded')
    OUTPUT_DIR = Path('/mnt/data/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE_outputs')
    DRIVE_OUTPUT_DIR = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_OUTPUT_DIR is not None:
    # This is a light operation. If it fails due to Drive connection, the notebook can still save locally.
    try:
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print('WARNING: Could not create DRIVE_OUTPUT_DIR now. Will retry at final ZIP copy step.')
        print(repr(e))


# Early save helper: defined here because carry-forward audit below writes outputs before the main helper section.
def save_df(df, filename):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print('saved', path, getattr(df, 'shape', ''))
    return path


def locate_file(candidates, preferred_dir=None, search_roots=None):
    """Locate an input CSV robustly.

    1. Try each candidate name in preferred_dir.
    2. If not found, search broader roots such as /content/drive.
    3. If multiple matches are found, prefer paths under a stage1 directory; otherwise stop with a clear error.
    """
    preferred_dir = Path(preferred_dir) if preferred_dir is not None else None
    candidates = list(candidates)

    if preferred_dir is not None:
        for name in candidates:
            p = preferred_dir / name
            if p.exists():
                print('FOUND in preferred_dir:', p)
                return p

    if search_roots is None:
        search_roots = [Path('/content/drive')] if IN_COLAB else [Path('/mnt/data')]

    matches = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        for name in candidates:
            try:
                matches.extend(root.rglob(name))
            except Exception as e:
                print('WARNING: search failed under', root, 'for', name, repr(e))

    # Remove duplicates while preserving order
    unique = []
    seen = set()
    for m in matches:
        s = str(m)
        if s not in seen:
            unique.append(m)
            seen.add(s)

    if len(unique) == 1:
        print('FOUND by broad search:', unique[0])
        return unique[0]

    if len(unique) > 1:
        stage1_like = [m for m in unique if 'stage1' in str(m).lower()]
        if len(stage1_like) == 1:
            print('FOUND by broad search, selected stage1-like path:', stage1_like[0])
            return stage1_like[0]
        print('Multiple candidate files found:')
        for i, m in enumerate(unique):
            print(f'{i}: {m}')
        raise FileExistsError('Multiple possible input files found. Set the path manually in the notebook.')

    raise FileNotFoundError(
        'Could not find any candidate file. Tried preferred_dir=' + str(preferred_dir) + '\n' +
        'Candidates:\n' + '\n'.join(candidates) + '\n' +
        'If the files are in Shared with me, copy them into MyDrive or add a Drive shortcut that Colab can access.'
    )

ONEYEAR_PATH = locate_file([
    'stage1_oneyear_mortality.csv',
    'stage1_oneyear_analysis.csv',
    'stage1_one_year_mortality.csv',
    'one_year_mortality_analysis.csv',
], preferred_dir=PREFERRED_STAGE1_DIR)

SURVIVAL_PATH = locate_file([
    'stage1_survival_analysis.csv',
    'stage1_survival.csv',
    'survival_analysis.csv',
    'stage1_full_followup_survival.csv',
], preferred_dir=PREFERRED_STAGE1_DIR)

HOSP_PATH = locate_file([
    'stage1_hospitalization_analysis.csv',
    'stage1_hospitalization.csv',
    'stage1_hosp.csv',
    'hospitalization_analysis.csv',
    'stage1_hosp_analysis.csv',
], preferred_dir=PREFERRED_STAGE1_DIR)

print('\nInput paths:')
print('ONEYEAR_PATH:', ONEYEAR_PATH)
print('SURVIVAL_PATH:', SURVIVAL_PATH)
print('HOSP_PATH:', HOSP_PATH)
print('\nLocal output dir:', OUTPUT_DIR)
print('Drive output dir:', DRIVE_OUTPUT_DIR)


FOUND in preferred_dir: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_oneyear_mortality.csv
FOUND in preferred_dir: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_survival_analysis.csv
FOUND in preferred_dir: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_hospitalization_analysis.csv

Input paths:
ONEYEAR_PATH: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_oneyear_mortality.csv
SURVIVAL_PATH: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_survival_analysis.csv
HOSP_PATH: /content/drive/MyDrive/dialysis/outputs/params/stage1/stage1_hospitalization_analysis.csv

Local output dir: /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE
Drive output dir: /content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE


## 1B. Carry-forward audit from Stage 5 primary outputs

This optional section reads selected outputs from the already-run Stage 5 v1.8.1 model notebook.

It is included for interpretation discipline only:

- it records the landmark/time-dependent pattern;
- it records whether full-cohort hospitalization NB was flagged;
- it preserves the rule that full cohort remains primary and ≥6m is mandatory sensitivity.

If the v1.8.1 output directory is not present, this section will skip gracefully.


In [3]:
# ============================================================
# 1B. Optional carry-forward from Stage 5 v1.8.1 outputs
# ============================================================
if IN_COLAB:
    PRIMARY_OUTPUT_DIR = Path('/content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC111_v1_8_1_COMMITTEE_REVIEWED')
else:
    PRIMARY_OUTPUT_DIR = Path('/mnt/data/DEC111_v1_8_1_COMMITTEE_REVIEWED_outputs')

carry_rows = []
reporting_rules = [
    {
        'topic': 'primary_model',
        'locked_rule': 'C-minimal remains the primary clinically guided model',
        'interpretation': 'Sensitivity results cannot replace C-minimal post hoc.'
    },
    {
        'topic': 'hospitalization_estimand',
        'locked_rule': 'Full cohort remains primary; ≥6m cohort is mandatory sensitivity',
        'interpretation': 'If full-cohort NB is flagged, report full and ≥6m side-by-side; do not silently switch primary cohort.'
    },
    {
        'topic': 'time_dependent_pattern',
        'locked_rule': 'PH/landmark findings are interpreted as time-dependent signal',
        'interpretation': 'E/e′ and albumin may operate at different time windows; this is a biological/clinical interpretation, not a model-selection rule.'
    },
    {
        'topic': 'C_lite_C_full',
        'locked_rule': 'C-lite/C-full are supportive/exploratory',
        'interpretation': 'They test whether SPAP, LA size and TR support or modify the C-minimal story, not whether they replace it.'
    },
]
reporting_rules_df = pd.DataFrame(reporting_rules)
save_df(reporting_rules_df, 'stage5_sensitivity_reporting_rules_DEC119_v1_3.csv')

if PRIMARY_OUTPUT_DIR.exists():
    print('Found Stage 5 primary output directory:', PRIMARY_OUTPUT_DIR)

    landmark_path = PRIMARY_OUTPUT_DIR / 'stage5_cox_landmark_effects.csv'
    if landmark_path.exists():
        lm = pd.read_csv(landmark_path)
        # Keep key variables if present.
        key_terms = ['TissueDopplerEERatioSeptal', 'albumin-numeric result', 'LV_EF']
        if 'term' in lm.columns:
            lm_key = lm[lm['term'].astype(str).isin(key_terms)].copy()
        elif 'covariate' in lm.columns:
            lm_key = lm[lm['covariate'].astype(str).isin(key_terms)].copy()
        else:
            lm_key = lm.copy()
        lm_key['source_file'] = landmark_path.name
        lm_key.to_csv(OUTPUT_DIR / 'stage5_sensitivity_carryforward_landmark_key_terms_DEC119_v1_3.csv', index=False)
        print('saved carry-forward landmark key terms', lm_key.shape)

    hosp_decision_path = PRIMARY_OUTPUT_DIR / 'stage5_hospitalization_model_decision_table_COMMITTEE.csv'
    if hosp_decision_path.exists():
        hd = pd.read_csv(hosp_decision_path)
        hd.to_csv(OUTPUT_DIR / 'stage5_sensitivity_carryforward_hospitalization_decision_table_DEC119_v1_3.csv', index=False)
        print('saved carry-forward hospitalization decision table', hd.shape)

    inc_path = PRIMARY_OUTPUT_DIR / 'stage5_primary_incremental_decision_helper_COMMITTEE_REVIEWED.csv'
    if inc_path.exists():
        inc = pd.read_csv(inc_path)
        inc.to_csv(OUTPUT_DIR / 'stage5_sensitivity_carryforward_primary_incremental_decisions_DEC119_v1_3.csv', index=False)
        print('saved carry-forward primary incremental decisions', inc.shape)
else:
    print('Stage 5 primary output directory not found; skipping carry-forward audit:', PRIMARY_OUTPUT_DIR)


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_reporting_rules_DEC119_v1_3.csv (4, 3)
Found Stage 5 primary output directory: /content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC111_v1_8_1_COMMITTEE_REVIEWED
saved carry-forward landmark key terms (6, 18)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


saved carry-forward hospitalization decision table (6, 13)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


saved carry-forward primary incremental decisions (5, 9)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
# ============================================================
# 2. Variable definitions and locked sensitivity tiers
# ============================================================
PID_COL = 'patient_id'

ONEYEAR_EVENT_COL = 'died_1year'

SURV_TIME_COL = 'time_to_event_days'
SURV_EVENT_COL = 'event'

HOSP_COUNT_COL = 'hosp_total'
HOSP_FOLLOWUP_DAYS_COL = 'followup_days'
HOSP_FOLLOWUP_YEARS_COL = 'followup_years'
HOSP_LOG_OFFSET_COL = 'log_followup_years'

AGE = 'AgeAtFirstHFDate'
SEX = 'm/f'
ALBUMIN = 'albumin-numeric result'
CREATININE = 'creatinine-numeric result'
AFIB = 'AFIB_binary'
EF = 'LV_EF'
EE_SEPTAL = 'TissueDopplerEERatioSeptal'
SPAP = 'EstimatedSysPAPressure'
LA_SIZE = 'LACavitySize'
TR_GROUP = 'tricuspid_regurgitation_clin_grouped'

MODEL_B = [AGE, SEX, ALBUMIN, CREATININE, AFIB, EF]
C_MINIMAL = MODEL_B + [EE_SEPTAL]
C_LITE = C_MINIMAL + [SPAP, LA_SIZE]
C_FULL = C_LITE + [TR_GROUP]

# Sensitivity model without albumin; not a primary model.
MODEL_B_NO_ALB = [AGE, SEX, CREATININE, AFIB, EF]
C_MINIMAL_NO_ALB = MODEL_B_NO_ALB + [EE_SEPTAL]

PREDICTOR_SETS = {
    'Model_B': MODEL_B,
    'C_minimal_primary_reference': C_MINIMAL,
    'C_lite_supportive': C_LITE,
    'C_full_exploratory': C_FULL,
}

NO_ALBUMIN_SETS = {
    'Model_B_no_albumin_sensitivity': MODEL_B_NO_ALB,
    'C_minimal_no_albumin_sensitivity': C_MINIMAL_NO_ALB,
}

# Matched-N sensitivity comparisons. These are not primary confirmatory tests.
SENSITIVITY_COMPARISONS = [
    {
        'comparison': 'C_lite_vs_C_minimal',
        'reduced_tier': 'C_minimal_primary_reference',
        'full_tier': 'C_lite_supportive',
        'matched_tier': 'C_lite_supportive',
        'added_variables': [SPAP, LA_SIZE],
        'role': 'supportive_sensitivity',
    },
    {
        'comparison': 'C_full_vs_C_lite',
        'reduced_tier': 'C_lite_supportive',
        'full_tier': 'C_full_exploratory',
        'matched_tier': 'C_full_exploratory',
        'added_variables': [TR_GROUP],
        'role': 'exploratory_sensitivity',
    },
]

SIX_MONTH_DAYS = 182.625
ALPHA = 0.05

# Committee diagnostic thresholds for hospitalization count models.
NB_ALPHA_NEAR_ZERO_THRESHOLD = 1e-6
NB_PEARSON_DISPERSION_REVIEW_THRESHOLD = 2.0
POISSON_SCALE_DISPERSION_REVIEW_THRESHOLD = 4.0

print('Predictor sets:')
for k, v in PREDICTOR_SETS.items():
    print(k, len(v), v)

Predictor sets:
Model_B 6 ['AgeAtFirstHFDate', 'm/f', 'albumin-numeric result', 'creatinine-numeric result', 'AFIB_binary', 'LV_EF']
C_minimal_primary_reference 7 ['AgeAtFirstHFDate', 'm/f', 'albumin-numeric result', 'creatinine-numeric result', 'AFIB_binary', 'LV_EF', 'TissueDopplerEERatioSeptal']
C_lite_supportive 9 ['AgeAtFirstHFDate', 'm/f', 'albumin-numeric result', 'creatinine-numeric result', 'AFIB_binary', 'LV_EF', 'TissueDopplerEERatioSeptal', 'EstimatedSysPAPressure', 'LACavitySize']
C_full_exploratory 10 ['AgeAtFirstHFDate', 'm/f', 'albumin-numeric result', 'creatinine-numeric result', 'AFIB_binary', 'LV_EF', 'TissueDopplerEERatioSeptal', 'EstimatedSysPAPressure', 'LACavitySize', 'tricuspid_regurgitation_clin_grouped']


In [5]:
# ============================================================
# 3. Load data and preflight checks
# ============================================================
one_df = pd.read_csv(ONEYEAR_PATH)
surv_df = pd.read_csv(SURVIVAL_PATH)
hosp_df = pd.read_csv(HOSP_PATH)

datasets = {'oneyear': one_df, 'survival': surv_df, 'hospitalization': hosp_df}
print({k: v.shape for k, v in datasets.items()})

for name, df in datasets.items():
    if PID_COL not in df.columns:
        raise ValueError(f'{name}: missing {PID_COL}')
    if df[PID_COL].isna().any():
        raise ValueError(f'{name}: missing patient_id values')
    dup = int(df[PID_COL].duplicated().sum())
    if dup != 0:
        raise ValueError(f'{name}: duplicated patient_id rows = {dup}')

required_primary = {
    'oneyear': [PID_COL, ONEYEAR_EVENT_COL] + C_MINIMAL,
    'survival': [PID_COL, SURV_TIME_COL, SURV_EVENT_COL] + C_MINIMAL,
    'hospitalization': [PID_COL, HOSP_COUNT_COL, HOSP_FOLLOWUP_DAYS_COL, HOSP_FOLLOWUP_YEARS_COL, HOSP_LOG_OFFSET_COL] + C_MINIMAL,
}
for name, cols in required_primary.items():
    missing = [c for c in cols if c not in datasets[name].columns]
    if missing:
        raise ValueError(f'{name}: missing required C-minimal columns: {missing}')

# Optional broader echo predictors: do not block C-minimal primary but do block this sensitivity notebook.
for name, df in datasets.items():
    missing_broader = [c for c in [SPAP, LA_SIZE, TR_GROUP] if c not in df.columns]
    if missing_broader:
        raise ValueError(f'{name}: missing broader sensitivity columns: {missing_broader}')

# Outcome validity checks.
if not set(one_df[ONEYEAR_EVENT_COL].dropna().unique()).issubset({0, 1, 0.0, 1.0}):
    raise ValueError('died_1year contains values outside 0/1')
if not set(surv_df[SURV_EVENT_COL].dropna().unique()).issubset({0, 1, 0.0, 1.0}):
    raise ValueError('event contains values outside 0/1')
if (surv_df[SURV_TIME_COL].dropna() <= 0).any():
    raise ValueError('time_to_event_days must be positive')
if (hosp_df[HOSP_COUNT_COL].dropna() < 0).any():
    raise ValueError('hosp_total must be non-negative')
if (hosp_df[HOSP_FOLLOWUP_YEARS_COL].dropna() <= 0).any():
    raise ValueError('followup_years must be positive')

calc_log = np.log(hosp_df[HOSP_FOLLOWUP_YEARS_COL].astype(float))
max_abs_log_diff = float(np.nanmax(np.abs(calc_log - hosp_df[HOSP_LOG_OFFSET_COL].astype(float))))
print('max_abs_log_offset_diff =', max_abs_log_diff)
if max_abs_log_diff > 1e-6:
    raise ValueError('log_followup_years does not match log(followup_years)')

# Hospitalization sensitivity cohort.
hosp_6m_df = hosp_df[pd.to_numeric(hosp_df[HOSP_FOLLOWUP_DAYS_COL], errors='coerce') >= SIX_MONTH_DAYS].copy()
print('hosp full N:', len(hosp_df), 'hosp >=6m N:', len(hosp_6m_df))
print('Preflight checks passed.')

{'oneyear': (617, 31), 'survival': (644, 32), 'hospitalization': (644, 34)}
max_abs_log_offset_diff = 1.3322676295501878e-15
hosp full N: 644 hosp >=6m N: 452
Preflight checks passed.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# ============================================================
# 4. Helper functions
# ============================================================
def save_df(df, filename):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False)
    print('saved', path, df.shape)
    return path


def complete_case(df, outcome_cols, vars_):
    cols = [PID_COL] + list(outcome_cols) + list(vars_)
    d = df[cols].copy()
    before = len(d)
    d = d.dropna(subset=list(outcome_cols) + list(vars_)).copy()
    return d, {
        'n_initial': before,
        'n_complete': len(d),
        'n_lost': before - len(d),
        'pct_lost': (before - len(d)) / before * 100 if before else np.nan,
    }


def build_design(df, vars_, add_constant=True):
    Xraw = df[list(vars_)].copy()
    source_map = {}
    for c in Xraw.columns:
        if pd.api.types.is_numeric_dtype(Xraw[c]):
            Xraw[c] = pd.to_numeric(Xraw[c], errors='coerce')
    X = pd.get_dummies(Xraw, drop_first=True, dtype=float)
    for col in X.columns:
        src = None
        if col in vars_:
            src = col
        else:
            for v in vars_:
                if col.startswith(str(v) + '_'):
                    src = v
                    break
        source_map[col] = src or col
    if add_constant:
        X = sm.add_constant(X, has_constant='add')
        source_map['const'] = 'const'
    return X.astype(float), source_map


def safe_exp(x):
    try:
        return float(np.exp(x))
    except Exception:
        return np.nan


def tidy_sm_effects(result, model_name, tier, outcome, effect_label='effect', extra=None):
    params = result.params.copy()
    bse = result.bse.copy() if hasattr(result, 'bse') else pd.Series(np.nan, index=params.index)
    rows = []
    for term, beta in params.items():
        if term == 'alpha' or str(term).startswith('inflate_'):
            continue
        se = bse.get(term, np.nan)
        z = beta / se if pd.notna(se) and se != 0 else np.nan
        p = 2 * (1 - st.norm.cdf(abs(z))) if pd.notna(z) else np.nan
        row = {
            'outcome': outcome,
            'model_name': model_name,
            'tier': tier,
            'term': term,
            'estimate_log': beta,
            'se_log': se,
            effect_label: safe_exp(beta),
            'ci_lower': safe_exp(beta - 1.96 * se) if pd.notna(se) else np.nan,
            'ci_upper': safe_exp(beta + 1.96 * se) if pd.notna(se) else np.nan,
            'p_value': p,
        }
        if extra:
            row.update(extra)
        rows.append(row)
    return pd.DataFrame(rows)


def lrt_from_models(reduced, full):
    ll0 = float(reduced.llf if hasattr(reduced, 'llf') else reduced.log_likelihood_)
    ll1 = float(full.llf if hasattr(full, 'llf') else full.log_likelihood_)
    if hasattr(reduced, 'params'):
        df0 = int(len(reduced.params))
    else:
        df0 = int(len(reduced.params_))
    if hasattr(full, 'params'):
        df1 = int(len(full.params))
    else:
        df1 = int(len(full.params_))
    stat = max(0.0, 2 * (ll1 - ll0))
    df_diff = max(1, df1 - df0)
    p = st.chi2.sf(stat, df_diff)
    return {'ll_reduced': ll0, 'll_full': ll1, 'lr_stat': stat, 'df_diff': df_diff, 'lr_p_value': p}


def pearson_dispersion_count(y, mu, df_resid, alpha=None):
    if df_resid <= 0:
        return np.nan
    if alpha is None or pd.isna(alpha):
        var = mu
    else:
        var = mu + alpha * (mu ** 2)
    var = np.where(var <= 0, np.nan, var)
    return float(np.nansum(((y - mu) ** 2) / var) / df_resid)


def flag_nb_diagnostics(alpha=np.nan, pearson_nb=np.nan, pearson_poisson=np.nan, converged=True, error=None):
    flags = []
    if error:
        flags.append('model_fit_error')
    if converged is not True:
        flags.append('not_converged_review')
    if pd.notna(alpha) and float(alpha) <= NB_ALPHA_NEAR_ZERO_THRESHOLD:
        flags.append('alpha_near_zero_degenerated_to_poisson_review')
    if pd.notna(pearson_nb) and float(pearson_nb) > NB_PEARSON_DISPERSION_REVIEW_THRESHOLD:
        flags.append('residual_overdispersion_after_NB_review')
    if pd.notna(pearson_poisson) and float(pearson_poisson) > POISSON_SCALE_DISPERSION_REVIEW_THRESHOLD:
        flags.append('high_poisson_scale_overdispersion_reference')
    return ';'.join(flags) if flags else 'OK'


def sensitivity_status_from_p(p):
    if pd.notna(p) and p < ALPHA:
        return 'supportive_signal_exploratory_not_primary'
    if pd.notna(p):
        return 'no_supportive_incremental_signal'
    return 'not_evaluable'

## 5. Complete-case feasibility for sensitivity tiers

This repeats the feasibility check for C-lite and C-full. These sample sizes should be reported with every sensitivity result.

In [7]:
# ============================================================
# 5. Complete-case feasibility
# ============================================================
cc_rows = []
for dataset_name, df, outcome_cols in [
    ('oneyear', one_df, [ONEYEAR_EVENT_COL]),
    ('survival', surv_df, [SURV_TIME_COL, SURV_EVENT_COL]),
    ('hospitalization_full', hosp_df, [HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL]),
    ('hospitalization_ge6m', hosp_6m_df, [HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL]),
]:
    for tier, vars_ in PREDICTOR_SETS.items():
        _, info = complete_case(df, outcome_cols, vars_)
        cc_rows.append({'dataset': dataset_name, 'tier': tier, **info})

cc_summary = pd.DataFrame(cc_rows)
display(cc_summary)
save_df(cc_summary, 'stage5_sensitivity_complete_case_feasibility.csv')

,dataset,tier,n_initial,n_complete,n_lost,pct_lost
0,oneyear,Model_B,617,612,5,0.810373
1,oneyear,C_minimal_primary_reference,617,497,120,19.448947
2,oneyear,C_lite_supportive,617,398,219,35.494327
3,oneyear,C_full_exploratory,617,391,226,36.628849
4,survival,Model_B,644,639,5,0.776398
5,survival,C_minimal_primary_reference,644,518,126,19.565217
6,survival,C_lite_supportive,644,415,229,35.559006
7,survival,C_full_exploratory,644,408,236,36.645963
8,hospitalization_full,Model_B,644,639,5,0.776398
9,hospitalization_full,C_minimal_primary_reference,644,518,126,19.565217


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_complete_case_feasibility.csv (16, 6)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_complete_case_feasibility.csv')

## 6. Logistic sensitivity: C-lite and C-full

Compares broader echo tiers using matched-N nested comparisons:

- C-lite vs C-minimal on the C-lite complete-case sample
- C-full vs C-lite on the C-full complete-case sample

These are supportive/exploratory and do not replace the primary C-minimal result.

In [8]:
# ============================================================
# 6. Logistic sensitivity models
# ============================================================
def fit_logistic_on_df(d, outcome_col, vars_):
    y = pd.to_numeric(d[outcome_col], errors='coerce').astype(int)
    X, _ = build_design(d, vars_, add_constant=True)
    res = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    auc = roc_auc_score(y, res.predict(X)) if y.nunique() == 2 else np.nan
    return res, X, auc

logit_sens_effects = []
logit_sens_models = []
for tier in ['C_minimal_primary_reference', 'C_lite_supportive', 'C_full_exploratory']:
    vars_ = PREDICTOR_SETS[tier]
    try:
        d, info = complete_case(one_df, [ONEYEAR_EVENT_COL], vars_)
        res, X, auc = fit_logistic_on_df(d, ONEYEAR_EVENT_COL, vars_)
        logit_sens_effects.append(tidy_sm_effects(res, 'logistic', tier, 'one_year_mortality', effect_label='OR', extra={'sample_label':'own_complete_case'}))
        logit_sens_models.append({'outcome':'one_year_mortality','model_name':'logistic','tier':tier,'sample_label':'own_complete_case',**info,'events':int(d[ONEYEAR_EVENT_COL].sum()),'llf':float(res.llf),'aic':float(res.aic),'auc_c_stat':float(auc),'converged':True})
    except Exception as e:
        logit_sens_models.append({'outcome':'one_year_mortality','model_name':'logistic','tier':tier,'sample_label':'own_complete_case','error':repr(e)})

logit_lrt_rows = []
for comp in SENSITIVITY_COMPARISONS:
    reduced_vars = PREDICTOR_SETS[comp['reduced_tier']]
    full_vars = PREDICTOR_SETS[comp['full_tier']]
    matched_vars = PREDICTOR_SETS[comp['matched_tier']]
    d, info = complete_case(one_df, [ONEYEAR_EVENT_COL], matched_vars)
    try:
        red, X_red, auc_red = fit_logistic_on_df(d, ONEYEAR_EVENT_COL, reduced_vars)
        full, X_full, auc_full = fit_logistic_on_df(d, ONEYEAR_EVENT_COL, full_vars)
        lr = lrt_from_models(red, full)
        lr.update({
            'outcome':'one_year_mortality','model_family':'logistic','comparison':comp['comparison'],
            'role':comp['role'],'matched_tier':comp['matched_tier'],'n_matched':len(d),'events':int(d[ONEYEAR_EVENT_COL].sum()),
            'added_variables':' + '.join(comp['added_variables']),
            'aic_reduced':float(red.aic),'aic_full':float(full.aic),'delta_aic_full_minus_reduced':float(full.aic - red.aic),
            'auc_reduced':float(auc_red),'auc_full':float(auc_full),'delta_auc_full_minus_reduced':float(auc_full - auc_red),
            'committee_status':sensitivity_status_from_p(lr.get('lr_p_value', np.nan)),
        })
        logit_lrt_rows.append(lr)
    except Exception as e:
        logit_lrt_rows.append({'outcome':'one_year_mortality','model_family':'logistic','comparison':comp['comparison'],'role':comp['role'],'error':repr(e),'committee_status':'model_fit_error_review'})

logit_sens_effects_df = pd.concat(logit_sens_effects, ignore_index=True) if logit_sens_effects else pd.DataFrame()
logit_sens_models_df = pd.DataFrame(logit_sens_models)
logit_sens_lrt_df = pd.DataFrame(logit_lrt_rows)

display(logit_sens_models_df)
display(logit_sens_lrt_df)
save_df(logit_sens_effects_df, 'stage5_sensitivity_logistic_effects_C_lite_C_full.csv')
save_df(logit_sens_models_df, 'stage5_sensitivity_logistic_model_summary_C_lite_C_full.csv')
save_df(logit_sens_lrt_df, 'stage5_sensitivity_logistic_LRT_C_lite_C_full_matchedN.csv')

,outcome,model_name,tier,sample_label,n_initial,n_complete,n_lost,pct_lost,events,llf,aic,auc_c_stat,converged
0,one_year_mortality,logistic,C_minimal_primary_reference,own_complete_case,617,497,120,19.448947,185,-280.133982,576.267964,0.760326,True
1,one_year_mortality,logistic,C_lite_supportive,own_complete_case,617,398,219,35.494327,164,-235.638045,495.276090,0.741661,True
2,one_year_mortality,logistic,C_full_exploratory,own_complete_case,617,391,226,36.628849,163,-229.119618,488.239236,0.749166,True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,ll_reduced,ll_full,lr_stat,df_diff,lr_p_value,outcome,model_family,comparison,role,matched_tier,n_matched,events,added_variables,aic_reduced,aic_full,delta_aic_full_minus_reduced,auc_reduced,auc_full,delta_auc_full_minus_reduced,committee_status
0,-236.710971,-235.638045,2.145853,4,0.708954,one_year_mortality,logistic,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,398,164,EstimatedSysPAPressure + LACavitySize,489.421943,495.276090,5.854147,0.738222,0.741661,0.003440,no_supportive_incremental_signal
1,-232.200659,-229.119618,6.162081,3,0.103985,one_year_mortality,logistic,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,391,163,tricuspid_regurgitation_clin_grouped,488.401318,488.239236,-0.162081,0.739748,0.749166,0.009418,no_supportive_incremental_signal


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_logistic_effects_C_lite_C_full.csv (35, 11)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_logistic_model_summary_C_lite_C_full.csv (3, 13)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_logistic_LRT_C_lite_C_full_matchedN.csv (2, 20)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_logistic_LRT_C_lite_C_full_matchedN.csv')

## 7. Survival sensitivity: regular Cox + PH diagnostics

Runs regular Cox for C-minimal/C-lite/C-full and matched-N nested comparisons. PH diagnostics are summarized by source variable.

In [9]:
# ============================================================
# 7A. Survival sensitivity: regular Cox + PH
# ============================================================
def fit_cox_on_df(d, time_col, event_col, vars_):
    X, src = build_design(d, vars_, add_constant=False)
    cdf = pd.concat([d[[PID_COL, time_col, event_col]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    cph = CoxPHFitter()
    cph.fit(cdf.drop(columns=[PID_COL]), duration_col=time_col, event_col=event_col, show_progress=False)
    return cph, cdf, src

cox_sens_effects = []
cox_sens_models = []
cox_ph_terms = []
cox_ph_sources = []

for tier in ['C_minimal_primary_reference', 'C_lite_supportive', 'C_full_exploratory']:
    vars_ = PREDICTOR_SETS[tier]
    try:
        d, info = complete_case(surv_df, [SURV_TIME_COL, SURV_EVENT_COL], vars_)
        cph, cdf, src = fit_cox_on_df(d, SURV_TIME_COL, SURV_EVENT_COL, vars_)
        summ = cph.summary.reset_index().rename(columns={'covariate':'term'})
        summ['outcome'] = 'survival'; summ['model_name'] = 'cox_regular'; summ['tier'] = tier; summ['sample_label']='own_complete_case'
        cox_sens_effects.append(summ)
        cox_sens_models.append({'outcome':'survival','model_name':'cox_regular','tier':tier,'sample_label':'own_complete_case',**info,'events':int(d[SURV_EVENT_COL].sum()),'llf':float(cph.log_likelihood_),'aic_partial':float(cph.AIC_partial_),'harrell_c':float(cph.concordance_index_),'converged':True})
        ph = proportional_hazard_test(cph, cdf.drop(columns=[PID_COL]), time_transform='rank')
        ph_df = ph.summary.reset_index().rename(columns={'index':'term'})
        ph_df['source_variable'] = ph_df['term'].map(src).fillna(ph_df['term'])
        ph_df['tier'] = tier; ph_df['sample_label']='own_complete_case'
        ph_df['ph_violation_p_lt_0_05'] = ph_df['p'] < ALPHA
        cox_ph_terms.append(ph_df)
        ph_src = ph_df.groupby(['tier','source_variable'], as_index=False).agg(min_ph_p=('p','min'), any_ph_violation=('ph_violation_p_lt_0_05','max'), n_terms=('term','nunique'))
        cox_ph_sources.append(ph_src)
    except Exception as e:
        cox_sens_models.append({'outcome':'survival','model_name':'cox_regular','tier':tier,'sample_label':'own_complete_case','error':repr(e)})

cox_lrt_rows = []
for comp in SENSITIVITY_COMPARISONS:
    reduced_vars = PREDICTOR_SETS[comp['reduced_tier']]
    full_vars = PREDICTOR_SETS[comp['full_tier']]
    matched_vars = PREDICTOR_SETS[comp['matched_tier']]
    d, info = complete_case(surv_df, [SURV_TIME_COL, SURV_EVENT_COL], matched_vars)
    try:
        red, _, _ = fit_cox_on_df(d, SURV_TIME_COL, SURV_EVENT_COL, reduced_vars)
        full, _, _ = fit_cox_on_df(d, SURV_TIME_COL, SURV_EVENT_COL, full_vars)
        lr = lrt_from_models(red, full)
        lr.update({
            'outcome':'survival','model_family':'cox_regular','comparison':comp['comparison'],
            'role':comp['role'],'matched_tier':comp['matched_tier'],'n_matched':len(d),'events':int(d[SURV_EVENT_COL].sum()),
            'added_variables':' + '.join(comp['added_variables']),
            'aic_reduced':float(red.AIC_partial_),'aic_full':float(full.AIC_partial_),'delta_aic_full_minus_reduced':float(full.AIC_partial_ - red.AIC_partial_),
            'harrell_c_reduced':float(red.concordance_index_),'harrell_c_full':float(full.concordance_index_),'delta_harrell_c_full_minus_reduced':float(full.concordance_index_ - red.concordance_index_),
            'committee_status':sensitivity_status_from_p(lr.get('lr_p_value', np.nan)),
        })
        cox_lrt_rows.append(lr)
    except Exception as e:
        cox_lrt_rows.append({'outcome':'survival','model_family':'cox_regular','comparison':comp['comparison'],'role':comp['role'],'error':repr(e),'committee_status':'model_fit_error_review'})

cox_sens_effects_df = pd.concat(cox_sens_effects, ignore_index=True) if cox_sens_effects else pd.DataFrame()
cox_sens_models_df = pd.DataFrame(cox_sens_models)
cox_ph_terms_df = pd.concat(cox_ph_terms, ignore_index=True) if cox_ph_terms else pd.DataFrame()
cox_ph_sources_df = pd.concat(cox_ph_sources, ignore_index=True) if cox_ph_sources else pd.DataFrame()
cox_sens_lrt_df = pd.DataFrame(cox_lrt_rows)

display(cox_sens_models_df)
display(cox_sens_lrt_df)
display(cox_ph_sources_df)
save_df(cox_sens_effects_df, 'stage5_sensitivity_cox_regular_effects_C_lite_C_full.csv')
save_df(cox_sens_models_df, 'stage5_sensitivity_cox_regular_model_summary_C_lite_C_full.csv')
save_df(cox_sens_lrt_df, 'stage5_sensitivity_cox_regular_LRT_C_lite_C_full_matchedN.csv')
save_df(cox_ph_terms_df, 'stage5_sensitivity_cox_PH_terms_C_lite_C_full.csv')
save_df(cox_ph_sources_df, 'stage5_sensitivity_cox_PH_by_source_variable_C_lite_C_full.csv')

,outcome,model_name,tier,sample_label,n_initial,n_complete,n_lost,pct_lost,events,llf,aic_partial,harrell_c,converged
0,survival,cox_regular,C_minimal_primary_reference,own_complete_case,644,518,126,19.565217,405,-2139.736515,4293.473030,0.689385,True
1,survival,cox_regular,C_lite_supportive,own_complete_case,644,415,229,35.559006,340,-1705.109214,3432.218428,0.689822,True
2,survival,cox_regular,C_full_exploratory,own_complete_case,644,408,236,36.645963,335,-1669.060839,3366.121678,0.694874,True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,ll_reduced,ll_full,lr_stat,df_diff,lr_p_value,outcome,model_family,comparison,role,matched_tier,n_matched,events,added_variables,aic_reduced,aic_full,delta_aic_full_minus_reduced,harrell_c_reduced,harrell_c_full,delta_harrell_c_full_minus_reduced,committee_status
0,-1706.445699,-1705.109214,2.672970,4,0.613953,survival,cox_regular,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,415,340,EstimatedSysPAPressure + LACavitySize,3426.891398,3432.218428,5.327030,0.687195,0.689822,0.002627,no_supportive_incremental_signal
1,-1674.291657,-1669.060839,10.461636,3,0.015023,survival,cox_regular,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,408,335,tricuspid_regurgitation_clin_grouped,3370.583314,3366.121678,-4.461636,0.689110,0.694874,0.005763,supportive_signal_exploratory_not_primary


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,tier,source_variable,min_ph_p,any_ph_violation,n_terms
0,C_minimal_primary_reference,AFIB_binary,0.988572,False,1
1,C_minimal_primary_reference,AgeAtFirstHFDate,0.174023,False,1
2,C_minimal_primary_reference,LV_EF,0.003167,True,1
3,C_minimal_primary_reference,TissueDopplerEERatioSeptal,0.019761,True,1
4,C_minimal_primary_reference,albumin-numeric result,0.000290,True,1
5,C_minimal_primary_reference,creatinine-numeric result,0.019927,True,1
6,C_minimal_primary_reference,m/f,0.805389,False,1
7,C_lite_supportive,AFIB_binary,0.995406,False,1
8,C_lite_supportive,AgeAtFirstHFDate,0.566645,False,1
9,C_lite_supportive,EstimatedSysPAPressure,0.736644,False,1


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_regular_effects_C_lite_C_full.csv (32, 16)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_regular_model_summary_C_lite_C_full.csv (3, 13)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_regular_LRT_C_lite_C_full_matchedN.csv (2, 20)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_PH_terms_C_lite_C_full.csv (32, 8)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_PH_by_source_variable_C_lite_C_full.csv (26, 5)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_PH_by_source_variable_C_lite_C_full.csv')

## 7B. Survival PH sensitivity: albumin time-varying coefficient for broader tiers

This applies the same albumin time-varying approach used in the primary notebook. Results are sensitivity/supportive only.

In [10]:
# ============================================================
# 7B. Cox time-varying albumin sensitivity for broader tiers
# ============================================================
def make_time_varying_dataset(base_df, time_col, event_col, vars_, split_times=None):
    d, info = complete_case(base_df, [time_col, event_col], vars_)
    X, src = build_design(d, vars_, add_constant=False)
    base = pd.concat([d[[PID_COL, time_col, event_col]].reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    if ALBUMIN not in base.columns:
        raise ValueError('Albumin column not found after design matrix construction; expected numeric albumin column.')
    times = base[time_col].astype(float).values
    events = base[event_col].astype(int).values
    if split_times is None:
        split_times = np.sort(np.unique(times[events == 1]))
    rows = []
    eps = 1e-6
    static_cols = [c for c in base.columns if c not in [time_col, event_col]]
    for _, row in base.iterrows():
        t = float(row[time_col])
        ev = int(row[event_col])
        cuts = split_times[(split_times > eps) & (split_times < t)]
        stops = list(cuts) + [t]
        start = 0.0
        for stop in stops:
            out = {c: row[c] for c in static_cols}
            out['start'] = max(start, 0.0)
            out['stop'] = float(stop)
            out['event_tv'] = int(ev == 1 and abs(stop - t) < 1e-9)
            out['albumin_x_log_time'] = float(row[ALBUMIN]) * float(np.log(max(stop, 1.0)))
            rows.append(out)
            start = float(stop)
    tv = pd.DataFrame(rows)
    tv = tv[tv['stop'] > tv['start']].copy()
    return tv, info


def fit_tv_albumin_on_fixed_df(d, time_col, event_col, vars_, tier):
    split_times = np.sort(np.unique(d.loc[d[event_col] == 1, time_col].astype(float).values))
    tv, info = make_time_varying_dataset(d, time_col, event_col, vars_, split_times=split_times)
    ctv = CoxTimeVaryingFitter()
    covariate_cols = [c for c in tv.columns if c not in [PID_COL, 'start', 'stop', 'event_tv']]
    ctv.fit(tv[[PID_COL, 'start', 'stop', 'event_tv'] + covariate_cols], id_col=PID_COL, start_col='start', stop_col='stop', event_col='event_tv', show_progress=False)
    eff = ctv.summary.reset_index().rename(columns={'covariate':'term'})
    eff['outcome'] = 'survival'; eff['model_name'] = 'cox_time_varying_albumin'; eff['tier'] = tier
    summary = {'outcome':'survival','model_name':'cox_time_varying_albumin','tier':tier,'n_subjects':int(d[PID_COL].nunique()),'n_intervals':int(len(tv)),'events':int(d[event_col].sum()),'llf':float(ctv.log_likelihood_),'n_params':int(len(ctv.params_)),'converged':True}
    return ctv, eff, summary

cox_tv_effects = []
cox_tv_models = []
cox_tv_lrt_rows = []
for comp in SENSITIVITY_COMPARISONS:
    reduced_vars = PREDICTOR_SETS[comp['reduced_tier']]
    full_vars = PREDICTOR_SETS[comp['full_tier']]
    matched_vars = PREDICTOR_SETS[comp['matched_tier']]
    d, info = complete_case(surv_df, [SURV_TIME_COL, SURV_EVENT_COL], matched_vars)
    try:
        red, red_eff, red_sum = fit_tv_albumin_on_fixed_df(d, SURV_TIME_COL, SURV_EVENT_COL, reduced_vars, comp['reduced_tier'] + '_matched_to_' + comp['matched_tier'])
        full, full_eff, full_sum = fit_tv_albumin_on_fixed_df(d, SURV_TIME_COL, SURV_EVENT_COL, full_vars, comp['full_tier'] + '_matched')
        cox_tv_effects.extend([red_eff, full_eff])
        cox_tv_models.extend([red_sum, full_sum])
        lr = lrt_from_models(red, full)
        lr.update({'outcome':'survival','model_family':'cox_time_varying_albumin','comparison':comp['comparison'],'role':comp['role'],'matched_tier':comp['matched_tier'],'n_matched':len(d),'events':int(d[SURV_EVENT_COL].sum()),'added_variables':' + '.join(comp['added_variables']),'committee_status':sensitivity_status_from_p(lr.get('lr_p_value', np.nan))})
        cox_tv_lrt_rows.append(lr)
    except Exception as e:
        cox_tv_lrt_rows.append({'outcome':'survival','model_family':'cox_time_varying_albumin','comparison':comp['comparison'],'role':comp['role'],'error':repr(e),'committee_status':'model_fit_error_review'})

cox_tv_effects_df = pd.concat(cox_tv_effects, ignore_index=True) if cox_tv_effects else pd.DataFrame()
cox_tv_models_df = pd.DataFrame(cox_tv_models)
cox_tv_lrt_df = pd.DataFrame(cox_tv_lrt_rows)

display(cox_tv_lrt_df)
save_df(cox_tv_effects_df, 'stage5_sensitivity_cox_timevarying_albumin_effects_C_lite_C_full.csv')
save_df(cox_tv_models_df, 'stage5_sensitivity_cox_timevarying_albumin_summary_C_lite_C_full.csv')
save_df(cox_tv_lrt_df, 'stage5_sensitivity_cox_timevarying_albumin_LRT_C_lite_C_full_matchedN.csv')

,ll_reduced,ll_full,lr_stat,df_diff,lr_p_value,outcome,model_family,comparison,role,matched_tier,n_matched,events,added_variables,committee_status
0,-1702.811941,-1701.284340,3.055203,4,0.548630,survival,cox_time_varying_albumin,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,415,340,EstimatedSysPAPressure + LACavitySize,no_supportive_incremental_signal
1,-1670.107242,-1665.250572,9.713341,3,0.021167,survival,cox_time_varying_albumin,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,408,335,tricuspid_regurgitation_clin_grouped,supportive_signal_exploratory_not_primary


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_timevarying_albumin_effects_C_lite_C_full.csv (47, 15)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_timevarying_albumin_summary_C_lite_C_full.csv (4, 9)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_timevarying_albumin_LRT_C_lite_C_full_matchedN.csv (2, 14)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_cox_timevarying_albumin_LRT_C_lite_C_full_matchedN.csv')

## 8. Hospitalization sensitivity: C-lite and C-full

Runs NB, robust Poisson and ZINB for full cohort and ≥6m cohort. Matched-N nested LRTs are shown for NB only, with diagnostic flags.

In [11]:
# ============================================================
# 8A. Hospitalization fitters
# ============================================================
def fit_poisson(df, count_col, offset_col, vars_, tier, cohort_label):
    d, info = complete_case(df, [count_col, offset_col], vars_)
    y = pd.to_numeric(d[count_col], errors='coerce').astype(float)
    X, _ = build_design(d, vars_, add_constant=True)
    offset = pd.to_numeric(d[offset_col], errors='coerce').astype(float)
    res = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit(cov_type='HC0')
    mu = res.predict(X, offset=offset)
    effects = tidy_sm_effects(res, 'poisson_robust', tier, 'hospitalization', effect_label='IRR', extra={'cohort_label':cohort_label})
    summary = {'outcome':'hospitalization','model_name':'poisson_robust','tier':tier,'cohort_label':cohort_label,**info,'events_total':float(y.sum()),'zero_count_n':int((y==0).sum()),'zero_count_pct':float((y==0).mean()*100),'llf':float(res.llf),'aic':float(res.aic),'alpha':np.nan,'pearson_dispersion_poisson':pearson_dispersion_count(y, mu, res.df_resid, alpha=None),'converged':True}
    return res, d, effects, summary


def fit_nb(df, count_col, offset_col, vars_, tier, cohort_label):
    d, info = complete_case(df, [count_col, offset_col], vars_)
    y = pd.to_numeric(d[count_col], errors='coerce').astype(float)
    X, _ = build_design(d, vars_, add_constant=True)
    offset = pd.to_numeric(d[offset_col], errors='coerce').astype(float)
    res = NegativeBinomial(y, X, offset=offset).fit(disp=0, maxiter=300)
    alpha = float(res.params.get('alpha', np.nan)) if hasattr(res.params, 'get') else np.nan
    beta_params = res.params.drop(labels=['alpha'], errors='ignore')
    mu = res.predict(X)
    k_params = len(beta_params)
    df_resid = len(d) - k_params
    pearson_nb = pearson_dispersion_count(y, mu, df_resid, alpha=alpha)
    pearson_poisson = pearson_dispersion_count(y, mu, df_resid, alpha=None)
    converged = bool(res.mle_retvals.get('converged', False))
    effects = tidy_sm_effects(res, 'negative_binomial', tier, 'hospitalization', effect_label='IRR', extra={'cohort_label':cohort_label})
    summary = {'outcome':'hospitalization','model_name':'negative_binomial','tier':tier,'cohort_label':cohort_label,**info,'events_total':float(y.sum()),'zero_count_n':int((y==0).sum()),'zero_count_pct':float((y==0).mean()*100),'llf':float(res.llf),'aic':float(res.aic),'bic':float(res.bic),'alpha':alpha,'pearson_dispersion_nb':pearson_nb,'pearson_dispersion_poisson_scale':pearson_poisson,'converged':converged,'warnflag':res.mle_retvals.get('warnflag', np.nan)}
    summary['nb_fit_flags'] = flag_nb_diagnostics(alpha=alpha, pearson_nb=pearson_nb, pearson_poisson=pearson_poisson, converged=converged)
    return res, d, effects, summary


def fit_zinb(df, count_col, offset_col, vars_, tier, cohort_label):
    d, info = complete_case(df, [count_col, offset_col], vars_)
    y = pd.to_numeric(d[count_col], errors='coerce').astype(float)
    X, _ = build_design(d, vars_, add_constant=True)
    offset = pd.to_numeric(d[offset_col], errors='coerce').astype(float)
    exog_infl = np.ones((len(d), 1))
    res = ZeroInflatedNegativeBinomialP(y, X, exog_infl=exog_infl, offset=offset, inflation='logit').fit(method='bfgs', maxiter=300, disp=0)
    alpha = float(res.params.get('alpha', np.nan)) if hasattr(res.params, 'get') else np.nan
    effects = tidy_sm_effects(res, 'zinb_intercept_inflation', tier, 'hospitalization', effect_label='IRR', extra={'cohort_label':cohort_label})
    summary = {'outcome':'hospitalization','model_name':'zinb_intercept_inflation','tier':tier,'cohort_label':cohort_label,**info,'events_total':float(y.sum()),'zero_count_n':int((y==0).sum()),'zero_count_pct':float((y==0).mean()*100),'llf':float(res.llf),'aic':float(res.aic),'bic':float(res.bic),'alpha':alpha,'converged':bool(res.mle_retvals.get('converged', False)),'warnflag':res.mle_retvals.get('warnflag', np.nan)}
    return res, d, effects, summary


def run_count_sensitivity_suite(df, cohort_label, tiers=('C_minimal_primary_reference','C_lite_supportive','C_full_exploratory')):
    effects=[]; summaries=[]; fits={}
    for tier in tiers:
        vars_ = PREDICTOR_SETS[tier]
        for fitter_name, fitter in [('negative_binomial', fit_nb), ('poisson_robust', fit_poisson), ('zinb_intercept_inflation', fit_zinb)]:
            try:
                res, d, eff, summ = fitter(df, HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL, vars_, tier, cohort_label)
                effects.append(eff); summaries.append(summ); fits[(tier, fitter_name)] = res
            except Exception as e:
                summaries.append({'outcome':'hospitalization','model_name':fitter_name,'tier':tier,'cohort_label':cohort_label,'error':repr(e)})
    return fits, (pd.concat(effects, ignore_index=True) if effects else pd.DataFrame()), pd.DataFrame(summaries)

fits_full, hosp_eff_full, hosp_sum_full = run_count_sensitivity_suite(hosp_df, 'full_cohort_primary')
fits_6m, hosp_eff_6m, hosp_sum_6m = run_count_sensitivity_suite(hosp_6m_df, 'observed_ge_6m_sensitivity')

hosp_sens_effects = pd.concat([hosp_eff_full, hosp_eff_6m], ignore_index=True, sort=False)
hosp_sens_summary = pd.concat([hosp_sum_full, hosp_sum_6m], ignore_index=True, sort=False)

display(hosp_sens_summary)
save_df(hosp_sens_effects, 'stage5_sensitivity_hospitalization_effects_all_models_C_lite_C_full.csv')
save_df(hosp_sens_summary, 'stage5_sensitivity_hospitalization_model_summary_all_models_C_lite_C_full.csv')

/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3375: RuntimeWarning: divide by zero encountered in scalar divide
  size = 1/alpha * mu**Q
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3376: RuntimeWarning: invalid value encountered in divide
  prob = size/(size+mu)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3377: RuntimeWarning: invalid value encountered in subtract
  coeff = (gamma_ln(size+endog) - gamma_ln(endog+1) -
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3455: Run

,outcome,model_name,tier,cohort_label,n_initial,n_complete,n_lost,pct_lost,events_total,zero_count_n,zero_count_pct,llf,aic,bic,alpha,pearson_dispersion_nb,pearson_dispersion_poisson_scale,converged,warnflag,nb_fit_flags,pearson_dispersion_poisson
0,hospitalization,negative_binomial,C_minimal_primary_reference,full_cohort_primary,644,518,126,19.565217,955.0,195,37.644788,-991.060873,2000.121747,2038.371524,1.496545e+00,2.096459,5.276193,True,0.0,residual_overdispersion_after_NB_review;high_p...,NaN
1,hospitalization,poisson_robust,C_minimal_primary_reference,full_cohort_primary,644,518,126,19.565217,955.0,195,37.644788,-1306.071787,2628.143573,NaN,NaN,NaN,NaN,True,NaN,NaN,6.861866
2,hospitalization,zinb_intercept_inflation,C_minimal_primary_reference,full_cohort_primary,644,518,126,19.565217,955.0,195,37.644788,-991.060873,2002.121747,2044.621499,1.496545e+00,NaN,NaN,True,0.0,NaN,NaN
3,hospitalization,negative_binomial,C_lite_supportive,full_cohort_primary,644,415,229,35.559006,781.0,161,38.795181,NaN,NaN,NaN,5.902614e-29,8.962240,8.962240,False,3.0,not_converged_review;alpha_near_zero_degenerat...,NaN
4,hospitalization,poisson_robust,C_lite_supportive,full_cohort_primary,644,415,229,35.559006,781.0,161,38.795181,-1072.196571,2168.393141,NaN,NaN,NaN,NaN,True,NaN,NaN,7.255548
5,hospitalization,zinb_intercept_inflation,C_lite_supportive,full_cohort_primary,644,415,229,35.559006,781.0,161,38.795181,-797.173988,1622.347975,1678.743875,1.563579e+00,NaN,NaN,True,0.0,NaN,NaN
6,hospitalization,negative_binomial,C_full_exploratory,full_cohort_primary,644,408,236,36.645963,766.0,160,39.215686,NaN,NaN,NaN,3.141036e-25,9.277721,9.277721,False,3.0,not_converged_review;alpha_near_zero_degenerat...,NaN
7,hospitalization,poisson_robust,C_full_exploratory,full_cohort_primary,644,408,236,36.645963,766.0,160,39.215686,-1049.102209,2128.204417,NaN,NaN,NaN,NaN,True,NaN,NaN,7.278592
8,hospitalization,zinb_intercept_inflation,C_full_exploratory,full_cohort_primary,644,408,236,36.645963,766.0,160,39.215686,NaN,NaN,NaN,-1.383317e+04,NaN,NaN,False,2.0,NaN,NaN
9,hospitalization,negative_binomial,C_minimal_primary_reference,observed_ge_6m_sensitivity,452,371,81,17.920354,874.0,96,25.876011,-827.426418,1672.852836,1708.098655,1.392884e+00,3.457027,7.806604,True,0.0,residual_overdispersion_after_NB_review;high_p...,NaN


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_hospitalization_effects_all_models_C_lite_C_full.csv (210, 11)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_hospitalization_model_summary_all_models_C_lite_C_full.csv (18, 21)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_hospitalization_model_summary_all_models_C_lite_C_full.csv')

In [12]:
# ============================================================
# 8B. Hospitalization matched-N NB LRTs with diagnostics
# ============================================================
def fit_nb_on_fixed_df(d, count_col, offset_col, vars_):
    X, _ = build_design(d, vars_, add_constant=True)
    y = pd.to_numeric(d[count_col], errors='coerce').astype(float)
    offset = pd.to_numeric(d[offset_col], errors='coerce').astype(float)
    res = NegativeBinomial(y, X, offset=offset).fit(disp=0, maxiter=300)
    return res, X


def nb_fixed_diagnostics(res, X, d, count_col):
    y = pd.to_numeric(d[count_col], errors='coerce').astype(float)
    beta_params = res.params.drop(labels=['alpha'], errors='ignore') if hasattr(res.params, 'drop') else res.params
    alpha = float(res.params.get('alpha', np.nan)) if hasattr(res.params, 'get') else np.nan
    mu = res.predict(X)
    df_resid = len(d) - len(beta_params)
    pearson_nb = pearson_dispersion_count(y, mu, df_resid, alpha=alpha)
    pearson_poisson = pearson_dispersion_count(y, mu, df_resid, alpha=None)
    converged = bool(res.mle_retvals.get('converged', False))
    flags = flag_nb_diagnostics(alpha=alpha, pearson_nb=pearson_nb, pearson_poisson=pearson_poisson, converged=converged)
    return {'alpha':alpha,'pearson_dispersion_nb':pearson_nb,'pearson_dispersion_poisson_scale':pearson_poisson,'converged':converged,'warnflag':res.mle_retvals.get('warnflag', np.nan),'nb_fit_flags':flags}

hosp_lrt_rows = []
for cohort_label, hdf in [('full_cohort_primary', hosp_df), ('observed_ge_6m_sensitivity', hosp_6m_df)]:
    for comp in SENSITIVITY_COMPARISONS:
        reduced_vars = PREDICTOR_SETS[comp['reduced_tier']]
        full_vars = PREDICTOR_SETS[comp['full_tier']]
        matched_vars = PREDICTOR_SETS[comp['matched_tier']]
        matched_h, info_h = complete_case(hdf, [HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL], matched_vars)
        try:
            nb_red, X_red = fit_nb_on_fixed_df(matched_h, HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL, reduced_vars)
            nb_full, X_full = fit_nb_on_fixed_df(matched_h, HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL, full_vars)
            diag_red = nb_fixed_diagnostics(nb_red, X_red, matched_h, HOSP_COUNT_COL)
            diag_full = nb_fixed_diagnostics(nb_full, X_full, matched_h, HOSP_COUNT_COL)
            lr = lrt_from_models(nb_red, nb_full)
            y = matched_h[HOSP_COUNT_COL].astype(float)
            flagged = (diag_red['nb_fit_flags'] != 'OK') or (diag_full['nb_fit_flags'] != 'OK')
            status = sensitivity_status_from_p(lr.get('lr_p_value', np.nan))
            if flagged and status.startswith('supportive_signal'):
                status = 'supportive_signal_but_NB_fit_flagged_interpret_with_sensitivity'
            lr.update({
                'outcome':'hospitalization','model_family':'negative_binomial','comparison':comp['comparison'],
                'role':comp['role'],'cohort_label':cohort_label,'matched_tier':comp['matched_tier'],'n_matched':len(matched_h),
                'events_total':float(y.sum()),'zero_count_n':int((y==0).sum()),'added_variables':' + '.join(comp['added_variables']),
                'aic_reduced':float(nb_red.aic),'aic_full':float(nb_full.aic),'delta_aic_full_minus_reduced':float(nb_full.aic - nb_red.aic),
                'alpha_reduced':diag_red['alpha'],'alpha_full':diag_full['alpha'],
                'pearson_dispersion_nb_reduced':diag_red['pearson_dispersion_nb'],'pearson_dispersion_nb_full':diag_full['pearson_dispersion_nb'],
                'pearson_dispersion_poisson_scale_reduced':diag_red['pearson_dispersion_poisson_scale'],'pearson_dispersion_poisson_scale_full':diag_full['pearson_dispersion_poisson_scale'],
                'converged_reduced':diag_red['converged'],'converged_full':diag_full['converged'],
                'nb_fit_flags_reduced':diag_red['nb_fit_flags'],'nb_fit_flags_full':diag_full['nb_fit_flags'],
                'committee_status':status,
            })
            hosp_lrt_rows.append(lr)
        except Exception as e:
            hosp_lrt_rows.append({'outcome':'hospitalization','model_family':'negative_binomial','comparison':comp['comparison'],'role':comp['role'],'cohort_label':cohort_label,'error':repr(e),'committee_status':'model_fit_error_review'})

hosp_sens_lrt_df = pd.DataFrame(hosp_lrt_rows)
display(hosp_sens_lrt_df)
save_df(hosp_sens_lrt_df, 'stage5_sensitivity_hospitalization_NB_LRT_C_lite_C_full_matchedN_FLAGGED.csv')

/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3375: RuntimeWarning: overflow encountered in scalar divide
  size = 1/alpha * mu**Q
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3376: RuntimeWarning: invalid value encountered in divide
  prob = size/(size+mu)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3377: RuntimeWarning: invalid value encountered in subtract
  coeff = (gamma_ln(size+endog) - gamma_ln(endog+1) -
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3455: RuntimeWa

,ll_reduced,ll_full,lr_stat,df_diff,lr_p_value,outcome,model_family,comparison,role,cohort_label,matched_tier,n_matched,events_total,zero_count_n,added_variables,aic_reduced,aic_full,delta_aic_full_minus_reduced,alpha_reduced,alpha_full,pearson_dispersion_nb_reduced,pearson_dispersion_nb_full,pearson_dispersion_poisson_scale_reduced,pearson_dispersion_poisson_scale_full,converged_reduced,converged_full,nb_fit_flags_reduced,nb_fit_flags_full,committee_status
0,NaN,NaN,0.000000,4,1.000000,hospitalization,negative_binomial,C_lite_vs_C_minimal,supportive_sensitivity,full_cohort_primary,C_lite_supportive,415,781.0,161,EstimatedSysPAPressure + LACavitySize,NaN,NaN,NaN,2.927428e-29,5.902614e-29,8.913833,8.962240,8.913833,8.962240,False,False,not_converged_review;alpha_near_zero_degenerat...,not_converged_review;alpha_near_zero_degenerat...,no_supportive_incremental_signal
1,NaN,NaN,0.000000,3,1.000000,hospitalization,negative_binomial,C_full_vs_C_lite,exploratory_sensitivity,full_cohort_primary,C_full_exploratory,408,766.0,160,tricuspid_regurgitation_clin_grouped,NaN,NaN,NaN,4.533954e-27,3.141036e-25,9.037820,9.277721,9.037820,9.277721,False,False,not_converged_review;alpha_near_zero_degenerat...,not_converged_review;alpha_near_zero_degenerat...,no_supportive_incremental_signal
2,-649.517677,-646.378658,6.278039,4,0.179324,hospitalization,negative_binomial,C_lite_vs_C_minimal,supportive_sensitivity,observed_ge_6m_sensitivity,C_lite_supportive,287,705.0,77,EstimatedSysPAPressure + LACavitySize,1317.035354,1318.757315,1.721961,1.485821e+00,1.444303e+00,2.995584,3.244954,7.444666,7.738926,True,True,residual_overdispersion_after_NB_review;high_p...,residual_overdispersion_after_NB_review;high_p...,no_supportive_incremental_signal
3,-633.388883,-632.247328,2.283111,3,0.515764,hospitalization,negative_binomial,C_full_vs_C_lite,exploratory_sensitivity,observed_ge_6m_sensitivity,C_full_exploratory,281,692.0,76,tricuspid_regurgitation_clin_grouped,1292.777767,1296.494656,3.716889,1.461057e+00,1.449682e+00,3.234427,3.455177,7.768118,8.047932,True,True,residual_overdispersion_after_NB_review;high_p...,residual_overdispersion_after_NB_review;high_p...,no_supportive_incremental_signal


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_hospitalization_NB_LRT_C_lite_C_full_matchedN_FLAGGED.csv (4, 29)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_hospitalization_NB_LRT_C_lite_C_full_matchedN_FLAGGED.csv')

## 9. No-albumin sensitivity

This is a **supportive sensitivity only**. It checks whether albumin may absorb part of the echo signal. It must not be used to replace the baseline clinical adjustment set.

In [13]:
# ============================================================
# 9. No-albumin sensitivity
# ============================================================
def extract_term_effect(df, term_contains, effect_col_candidates=('OR','HR','IRR','exp(coef)')):
    if df is None or len(df) == 0 or 'term' not in df.columns:
        return pd.DataFrame()
    d = df[df['term'].astype(str).str.contains(term_contains, regex=False)].copy()
    return d

noalb_rows_effects=[]
noalb_rows_models=[]

# Logistic no-albumin on C-minimal matched sample for comparison of E/e' effect.
try:
    d_with, _ = complete_case(one_df, [ONEYEAR_EVENT_COL], C_MINIMAL)
    # Use same rows, but reduced variables without albumin.
    res_noalb, X_noalb, auc_noalb = fit_logistic_on_df(d_with, ONEYEAR_EVENT_COL, C_MINIMAL_NO_ALB)
    res_with, X_with, auc_with = fit_logistic_on_df(d_with, ONEYEAR_EVENT_COL, C_MINIMAL)
    noalb_rows_effects.append(tidy_sm_effects(res_noalb, 'logistic_no_albumin', 'C_minimal_no_albumin_sensitivity', 'one_year_mortality', effect_label='OR', extra={'sample_label':'matched_to_C_minimal_with_albumin'}))
    noalb_rows_effects.append(tidy_sm_effects(res_with, 'logistic_with_albumin_reference', 'C_minimal_primary_reference', 'one_year_mortality', effect_label='OR', extra={'sample_label':'matched_to_C_minimal_with_albumin'}))
    noalb_rows_models.append({'outcome':'one_year_mortality','model_family':'logistic','comparison':'C_minimal_no_albumin_vs_with_albumin_descriptive','n_matched':len(d_with),'aic_no_albumin':float(res_noalb.aic),'aic_with_albumin':float(res_with.aic),'auc_no_albumin':float(auc_noalb),'auc_with_albumin':float(auc_with),'note':'descriptive sensitivity; not a replacement for primary model'})
except Exception as e:
    noalb_rows_models.append({'outcome':'one_year_mortality','model_family':'logistic','error':repr(e)})

# Cox no-albumin.
try:
    d_with, _ = complete_case(surv_df, [SURV_TIME_COL, SURV_EVENT_COL], C_MINIMAL)
    cph_noalb, _, _ = fit_cox_on_df(d_with, SURV_TIME_COL, SURV_EVENT_COL, C_MINIMAL_NO_ALB)
    cph_with, _, _ = fit_cox_on_df(d_with, SURV_TIME_COL, SURV_EVENT_COL, C_MINIMAL)
    eff_no = cph_noalb.summary.reset_index().rename(columns={'covariate':'term'})
    eff_no['outcome']='survival'; eff_no['model_name']='cox_regular_no_albumin'; eff_no['tier']='C_minimal_no_albumin_sensitivity'; eff_no['sample_label']='matched_to_C_minimal_with_albumin'
    eff_w = cph_with.summary.reset_index().rename(columns={'covariate':'term'})
    eff_w['outcome']='survival'; eff_w['model_name']='cox_regular_with_albumin_reference'; eff_w['tier']='C_minimal_primary_reference'; eff_w['sample_label']='matched_to_C_minimal_with_albumin'
    noalb_rows_effects.extend([eff_no, eff_w])
    noalb_rows_models.append({'outcome':'survival','model_family':'cox_regular','comparison':'C_minimal_no_albumin_vs_with_albumin_descriptive','n_matched':len(d_with),'aic_no_albumin':float(cph_noalb.AIC_partial_),'aic_with_albumin':float(cph_with.AIC_partial_),'harrell_c_no_albumin':float(cph_noalb.concordance_index_),'harrell_c_with_albumin':float(cph_with.concordance_index_),'note':'descriptive sensitivity; not a replacement for primary model'})
except Exception as e:
    noalb_rows_models.append({'outcome':'survival','model_family':'cox_regular','error':repr(e)})

# Hospitalization no-albumin NB full and 6m.
for cohort_label, hdf in [('full_cohort_primary', hosp_df), ('observed_ge_6m_sensitivity', hosp_6m_df)]:
    try:
        d_with, _ = complete_case(hdf, [HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL], C_MINIMAL)
        nb_noalb, X_no = fit_nb_on_fixed_df(d_with, HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL, C_MINIMAL_NO_ALB)
        nb_with, X_w = fit_nb_on_fixed_df(d_with, HOSP_COUNT_COL, HOSP_LOG_OFFSET_COL, C_MINIMAL)
        noalb_rows_effects.append(tidy_sm_effects(nb_noalb, 'negative_binomial_no_albumin', 'C_minimal_no_albumin_sensitivity', 'hospitalization', effect_label='IRR', extra={'cohort_label':cohort_label,'sample_label':'matched_to_C_minimal_with_albumin'}))
        noalb_rows_effects.append(tidy_sm_effects(nb_with, 'negative_binomial_with_albumin_reference', 'C_minimal_primary_reference', 'hospitalization', effect_label='IRR', extra={'cohort_label':cohort_label,'sample_label':'matched_to_C_minimal_with_albumin'}))
        diag_no = nb_fixed_diagnostics(nb_noalb, X_no, d_with, HOSP_COUNT_COL)
        diag_w = nb_fixed_diagnostics(nb_with, X_w, d_with, HOSP_COUNT_COL)
        noalb_rows_models.append({'outcome':'hospitalization','model_family':'negative_binomial','cohort_label':cohort_label,'comparison':'C_minimal_no_albumin_vs_with_albumin_descriptive','n_matched':len(d_with),'aic_no_albumin':float(nb_noalb.aic),'aic_with_albumin':float(nb_with.aic),'alpha_no_albumin':diag_no['alpha'],'alpha_with_albumin':diag_w['alpha'],'nb_flags_no_albumin':diag_no['nb_fit_flags'],'nb_flags_with_albumin':diag_w['nb_fit_flags'],'note':'descriptive sensitivity; not a replacement for primary model'})
    except Exception as e:
        noalb_rows_models.append({'outcome':'hospitalization','model_family':'negative_binomial','cohort_label':cohort_label,'error':repr(e)})

noalb_effects_df = pd.concat(noalb_rows_effects, ignore_index=True, sort=False) if noalb_rows_effects else pd.DataFrame()
noalb_models_df = pd.DataFrame(noalb_rows_models)

display(noalb_models_df)
save_df(noalb_effects_df, 'stage5_sensitivity_no_albumin_effects_descriptive.csv')
save_df(noalb_models_df, 'stage5_sensitivity_no_albumin_model_summary_descriptive.csv')

/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/usr/local/lib/python3.12/dist-packages/statsmodels/discrete/discrete_model.py:3375: RuntimeWarning: divide by zero encountered in scalar divide
  size = 1/alpha * mu**Q
/usr/local/lib/python3.12/dist-packages/statsmodels/di

,outcome,model_family,comparison,n_matched,aic_no_albumin,aic_with_albumin,auc_no_albumin,auc_with_albumin,note,harrell_c_no_albumin,harrell_c_with_albumin,cohort_label,alpha_no_albumin,alpha_with_albumin,nb_flags_no_albumin,nb_flags_with_albumin
0,one_year_mortality,logistic,C_minimal_no_albumin_vs_with_albumin_descriptive,497,590.957473,576.267964,0.740263,0.760326,descriptive sensitivity; not a replacement for...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,survival,cox_regular,C_minimal_no_albumin_vs_with_albumin_descriptive,518,4299.819988,4293.473030,NaN,NaN,descriptive sensitivity; not a replacement for...,0.680672,0.689385,NaN,NaN,NaN,NaN,NaN
2,hospitalization,negative_binomial,C_minimal_no_albumin_vs_with_albumin_descriptive,518,1999.272452,2000.121747,NaN,NaN,descriptive sensitivity; not a replacement for...,NaN,NaN,full_cohort_primary,1.503325,1.496545,residual_overdispersion_after_NB_review;high_p...,residual_overdispersion_after_NB_review;high_p...
3,hospitalization,negative_binomial,C_minimal_no_albumin_vs_with_albumin_descriptive,371,1671.924839,1672.852836,NaN,NaN,descriptive sensitivity; not a replacement for...,NaN,NaN,observed_ge_6m_sensitivity,1.398165,1.392884,residual_overdispersion_after_NB_review;high_p...,residual_overdispersion_after_NB_review;high_p...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_no_albumin_effects_descriptive.csv (58, 24)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_no_albumin_model_summary_descriptive.csv (4, 16)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_no_albumin_model_summary_descriptive.csv')

## 10. Committee sensitivity decision summaries

These summaries are designed to help interpretation. They do not turn exploratory results into primary findings.

In [14]:
# ============================================================
# 10. Decision summaries and review tables
# ============================================================
all_lrt = pd.concat([
    logit_sens_lrt_df.assign(section='logistic'),
    cox_sens_lrt_df.assign(section='cox_regular'),
    cox_tv_lrt_df.assign(section='cox_time_varying_albumin'),
    hosp_sens_lrt_df.assign(section='hospitalization_NB'),
], ignore_index=True, sort=False)

# Add interpretation guardrails.
all_lrt['interpretation_guardrail'] = all_lrt.apply(
    lambda r: 'supportive_or_exploratory_only; do_not_replace_primary_Cminimal' if pd.isna(r.get('error', np.nan)) else 'model_error_review',
    axis=1
)

# Focused echo term tables for review.
terms_to_review = [EE_SEPTAL, SPAP, LA_SIZE, TR_GROUP, EF]

def keep_echo_terms(df):
    if df is None or len(df) == 0 or 'term' not in df.columns:
        return pd.DataFrame()
    mask = pd.Series(False, index=df.index)
    for t in terms_to_review:
        mask = mask | df['term'].astype(str).str.contains(str(t), regex=False)
    return df[mask].copy()

echo_effects_review = pd.concat([
    keep_echo_terms(logit_sens_effects_df).assign(section='logistic'),
    keep_echo_terms(cox_sens_effects_df).assign(section='cox_regular'),
    keep_echo_terms(cox_tv_effects_df).assign(section='cox_time_varying_albumin'),
    keep_echo_terms(hosp_sens_effects).assign(section='hospitalization'),
    keep_echo_terms(noalb_effects_df).assign(section='no_albumin'),
], ignore_index=True, sort=False)

display(all_lrt)
display(echo_effects_review.head(50))
save_df(all_lrt, 'stage5_sensitivity_all_incremental_LRT_summary_COMMITTEE.csv')
save_df(echo_effects_review, 'stage5_sensitivity_echo_terms_effects_review_COMMITTEE.csv')

# Compact committee decision table.
decision_rows=[]
for _, r in all_lrt.iterrows():
    decision_rows.append({
        'section': r.get('section',''),
        'outcome': r.get('outcome',''),
        'comparison': r.get('comparison',''),
        'cohort_label': r.get('cohort_label',''),
        'role': r.get('role',''),
        'n_matched': r.get('n_matched', np.nan),
        'added_variables': r.get('added_variables',''),
        'lr_p_value': r.get('lr_p_value', np.nan),
        'delta_aic_full_minus_reduced': r.get('delta_aic_full_minus_reduced', np.nan),
        'committee_status': r.get('committee_status',''),
        'interpretation': 'supportive/exploratory only; check complete-case loss and diagnostics before citing in manuscript',
    })
committee_decision = pd.DataFrame(decision_rows)
display(committee_decision)
save_df(committee_decision, 'stage5_sensitivity_committee_decision_table.csv')

,ll_reduced,ll_full,lr_stat,df_diff,lr_p_value,outcome,model_family,comparison,role,matched_tier,n_matched,events,added_variables,aic_reduced,aic_full,delta_aic_full_minus_reduced,auc_reduced,auc_full,delta_auc_full_minus_reduced,committee_status,section,harrell_c_reduced,harrell_c_full,delta_harrell_c_full_minus_reduced,cohort_label,events_total,zero_count_n,alpha_reduced,alpha_full,pearson_dispersion_nb_reduced,pearson_dispersion_nb_full,pearson_dispersion_poisson_scale_reduced,pearson_dispersion_poisson_scale_full,converged_reduced,converged_full,nb_fit_flags_reduced,nb_fit_flags_full,interpretation_guardrail
0,-236.710971,-235.638045,2.145853,4,0.708954,one_year_mortality,logistic,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,398,164.0,EstimatedSysPAPressure + LACavitySize,489.421943,495.276090,5.854147,0.738222,0.741661,0.003440,no_supportive_incremental_signal,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
1,-232.200659,-229.119618,6.162081,3,0.103985,one_year_mortality,logistic,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,391,163.0,tricuspid_regurgitation_clin_grouped,488.401318,488.239236,-0.162081,0.739748,0.749166,0.009418,no_supportive_incremental_signal,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
2,-1706.445699,-1705.109214,2.672970,4,0.613953,survival,cox_regular,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,415,340.0,EstimatedSysPAPressure + LACavitySize,3426.891398,3432.218428,5.327030,NaN,NaN,NaN,no_supportive_incremental_signal,cox_regular,0.687195,0.689822,0.002627,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
3,-1674.291657,-1669.060839,10.461636,3,0.015023,survival,cox_regular,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,408,335.0,tricuspid_regurgitation_clin_grouped,3370.583314,3366.121678,-4.461636,NaN,NaN,NaN,supportive_signal_exploratory_not_primary,cox_regular,0.689110,0.694874,0.005763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
4,-1702.811941,-1701.284340,3.055203,4,0.548630,survival,cox_time_varying_albumin,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,415,340.0,EstimatedSysPAPressure + LACavitySize,NaN,NaN,NaN,NaN,NaN,NaN,no_supportive_incremental_signal,cox_time_varying_albumin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
5,-1670.107242,-1665.250572,9.713341,3,0.021167,survival,cox_time_varying_albumin,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,408,335.0,tricuspid_regurgitation_clin_grouped,NaN,NaN,NaN,NaN,NaN,NaN,supportive_signal_exploratory_not_primary,cox_time_varying_albumin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,supportive_or_exploratory_only; do_not_replace...
6,NaN,NaN,0.000000,4,1.000000,hospitalization,negative_binomial,C_lite_vs_C_minimal,supportive_sensitivity,C_lite_supportive,415,NaN,EstimatedSysPAPressure + LACavitySize,NaN,NaN,NaN,NaN,NaN,NaN,no_supportive_incremental_signal,hospitalization_NB,NaN,NaN,NaN,full_cohort_primary,781.0,161.0,2.927428e-29,5.902614e-29,8.913833,8.962240,8.913833,8.962240,False,False,not_converged_review;alpha_near_zero_degenerat...,not_converged_review;alpha_near_zero_degenerat...,supportive_or_exploratory_only; do_not_replace...
7,NaN,NaN,0.000000,3,1.000000,hospitalization,negative_binomial,C_full_vs_C_lite,exploratory_sensitivity,C_full_exploratory,408,NaN,tricuspid_regurgitation_clin_grouped,NaN,NaN,NaN,NaN,NaN,NaN,no_supportive_incremental_signal,hospitalization_NB,NaN,NaN,NaN,full_cohort_primary,766.0,160.0,4.533954e-27,3.141036e-25,9.037820,9.277721,9.037820,9.277721,False,False,not_converged_review;alpha_near_zero_degenerat...,not_converged_review;alpha_near_zero_degenerat...,supportive_or_exploratory

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,outcome,model_name,tier,term,estimate_log,se_log,OR,ci_lower,ci_upper,p_value,sample_label,section,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p),IRR,cohort_label
0,one_year_mortality,logistic,C_minimal_primary_reference,LV_EF,-0.032982,0.007201,0.967556,0.953995,0.981309,0.000005,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,one_year_mortality,logistic,C_minimal_primary_reference,TissueDopplerEERatioSeptal,0.010512,0.011252,1.010567,0.988523,1.033102,0.350215,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,one_year_mortality,logistic,C_lite_supportive,LV_EF,-0.030088,0.008085,0.970360,0.955105,0.985858,0.000198,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,one_year_mortality,logistic,C_lite_supportive,TissueDopplerEERatioSeptal,0.005466,0.012357,1.005481,0.981422,1.030130,0.658240,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,one_year_mortality,logistic,C_lite_supportive,EstimatedSysPAPressure,-0.005284,0.008549,0.994730,0.978201,1.011539,0.536559,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,one_year_mortality,logistic,C_lite_supportive,LACavitySize_Moderately dilated,0.345643,0.268034,1.412898,0.835517,2.389275,0.197208,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,one_year_mortality,logistic,C_lite_supportive,LACavitySize_Normal,0.279620,0.407993,1.322628,0.594493,2.942581,0.493120,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,one_year_mortality,logistic,C_lite_supportive,LACavitySize_Severely dilated,0.136983,0.469097,1.146809,0.457286,2.876036,0.770275,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,one_year_mortality,logistic,C_full_exploratory,LV_EF,-0.026674,0.008307,0.973679,0.957954,0.989661,0.001322,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,one_year_mortality,logistic,C_full_exploratory,TissueDopplerEERatioSeptal,0.007630,0.012870,1.007660,0.982559,1.033402,0.553266,own_complete_case,logistic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_all_incremental_LRT_summary_COMMITTEE.csv (10, 38)
saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_echo_terms_effects_review_COMMITTEE.csv (175, 25)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,section,outcome,comparison,cohort_label,role,n_matched,added_variables,lr_p_value,delta_aic_full_minus_reduced,committee_status,interpretation
0,logistic,one_year_mortality,C_lite_vs_C_minimal,NaN,supportive_sensitivity,398,EstimatedSysPAPressure + LACavitySize,0.708954,5.854147,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
1,logistic,one_year_mortality,C_full_vs_C_lite,NaN,exploratory_sensitivity,391,tricuspid_regurgitation_clin_grouped,0.103985,-0.162081,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
2,cox_regular,survival,C_lite_vs_C_minimal,NaN,supportive_sensitivity,415,EstimatedSysPAPressure + LACavitySize,0.613953,5.327030,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
3,cox_regular,survival,C_full_vs_C_lite,NaN,exploratory_sensitivity,408,tricuspid_regurgitation_clin_grouped,0.015023,-4.461636,supportive_signal_exploratory_not_primary,supportive/exploratory only; check complete-ca...
4,cox_time_varying_albumin,survival,C_lite_vs_C_minimal,NaN,supportive_sensitivity,415,EstimatedSysPAPressure + LACavitySize,0.548630,NaN,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
5,cox_time_varying_albumin,survival,C_full_vs_C_lite,NaN,exploratory_sensitivity,408,tricuspid_regurgitation_clin_grouped,0.021167,NaN,supportive_signal_exploratory_not_primary,supportive/exploratory only; check complete-ca...
6,hospitalization_NB,hospitalization,C_lite_vs_C_minimal,full_cohort_primary,supportive_sensitivity,415,EstimatedSysPAPressure + LACavitySize,1.000000,NaN,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
7,hospitalization_NB,hospitalization,C_full_vs_C_lite,full_cohort_primary,exploratory_sensitivity,408,tricuspid_regurgitation_clin_grouped,1.000000,NaN,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
8,hospitalization_NB,hospitalization,C_lite_vs_C_minimal,observed_ge_6m_sensitivity,supportive_sensitivity,287,EstimatedSysPAPressure + LACavitySize,0.179324,1.721961,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...
9,hospitalization_NB,hospitalization,C_full_vs_C_lite,observed_ge_6m_sensitivity,exploratory_sensitivity,281,tricuspid_regurgitation_clin_grouped,0.515764,3.716889,no_supportive_incremental_signal,supportive/exploratory only; check complete-ca...


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_committee_decision_table.csv (10, 11)


PosixPath('/content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_committee_decision_table.csv')

In [15]:
# ============================================================
# 11. Run metadata + create ZIP and copy to Drive
# ============================================================
metadata = {
    'run_version': RUN_VERSION,
    'run_timestamp': RUN_TS,
    'notebook_role': 'post-primary sensitivity/supportive analyses; not primary model selection',
    'one_year_path': str(ONEYEAR_PATH),
    'survival_path': str(SURVIVAL_PATH),
    'hospitalization_path': str(HOSP_PATH),
    'local_output_dir': str(OUTPUT_DIR),
    'drive_output_dir': str(DRIVE_OUTPUT_DIR) if DRIVE_OUTPUT_DIR is not None else None,
    'primary_model_reference': 'C_minimal = clinical baseline + EF + E/e_prime septal',
    'sensitivity_tiers': list(PREDICTOR_SETS.keys()),
    'comparisons': SENSITIVITY_COMPARISONS,
    'interpretation_rule': 'C-lite/C-full/no-albumin analyses are supportive/exploratory and do not replace the primary C-minimal analysis.',
    'drive_safe_note': 'Outputs were written locally first. A ZIP is copied to Drive at the end to reduce Google Drive connection-abort errors.',
}
meta_path = OUTPUT_DIR / 'stage5_sensitivity_run_metadata_DEC119_v1_3.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('saved', meta_path)

# Create a ZIP locally.
zip_base = Path('/content') / RUN_VERSION if IN_COLAB else OUTPUT_DIR.parent / RUN_VERSION
zip_path_str = shutil.make_archive(str(zip_base), 'zip', str(OUTPUT_DIR))
zip_path = Path(zip_path_str)
print('created local ZIP:', zip_path)

# Copy the ZIP to Drive. If Drive is temporarily disconnected, the local ZIP remains available for manual download.
if IN_COLAB and DRIVE_OUTPUT_DIR is not None:
    try:
        if not Path('/content/drive/MyDrive').exists():
            from google.colab import drive
            drive.mount('/content/drive', force_remount=True)
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        zip_dst = DRIVE_OUTPUT_DIR.parent / (RUN_VERSION + '.zip')
        shutil.copy2(zip_path, zip_dst)
        # Also copy metadata for quick inspection.
        shutil.copy2(meta_path, DRIVE_OUTPUT_DIR / meta_path.name)
        print('copied ZIP to Drive:', zip_dst)
        print('copied metadata to Drive:', DRIVE_OUTPUT_DIR / meta_path.name)
    except Exception as e:
        print('WARNING: Could not copy ZIP to Drive due to:')
        print(repr(e))
        print('The local ZIP is still available at:', zip_path)
        print('Use the Colab file browser to download it manually if needed.')

print('DONE: Stage 5 sensitivity notebook completed.')


saved /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_run_metadata_DEC119_v1_3.json
created local ZIP: /content/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE.zip
copied ZIP to Drive: /content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE.zip
copied metadata to Drive: /content/drive/MyDrive/dialysis/outputs/params/stage5/outputs/DEC119_v1_3_SENSITIVITY_MODELS_DRIVE_SAFE/stage5_sensitivity_run_metadata_DEC119_v1_3.json
DONE: Stage 5 sensitivity notebook completed.
